In [ ]:
# n_estimators = 1400,
#  min_samples_split = 10,
#  min_samples_leaf = 4,
#  max_features = "sqrt",
#  max_depth = None,
#  bootstrap = True)

#  max_depth=20, min_samples_leaf=4,
#                                         min_samples_split=5,
#                                         n_estimators=800


In [ ]:
#BAYES OPTIMIZATION
# max_depth=3, max_features='sqrt', min_samples_leaf=4,
#                        min_samples_split=4, n_estimators=473, n_jobs=2


In [ ]:
proba_valid = pipelinerf.predict_proba(x_val)[:, 1]

from sklearn.isotonic import IsotonicRegression
from sklearn.calibration import calibration_curve
iso_reg = IsotonicRegression(y_min = 0, y_max = 1, out_of_bounds = 'clip').fit(proba_valid, y_val)
proba_test_forest_isoreg = iso_reg.predict(pipelinerf.predict_proba(x_test)[:, 1])

print(proba_test_forest_isoreg)

proba_test_forest = pipelinerf.predict_proba(x_test)[:, 1]

def expected_calibration_error(y, proba, bins = 'fd'):
  import numpy as np
  bin_count, bin_edges = np.histogram(proba, bins = bins)
  n_bins = len(bin_count)
  bin_edges[0] -= 1e-8 # because left edge is not included
  bin_id = np.digitize(proba, bin_edges, right = True) - 1
  bin_ysum = np.bincount(bin_id, weights = y, minlength = n_bins)
  bin_probasum = np.bincount(bin_id, weights = proba, minlength = n_bins)
  bin_ymean = np.divide(bin_ysum, bin_count, out = np.zeros(n_bins), where = bin_count > 0)
  bin_probamean = np.divide(bin_probasum, bin_count, out = np.zeros(n_bins), where = bin_count > 0)
  ece = np.abs((bin_probamean - bin_ymean) * bin_count).sum() / len(proba)
  return ece